# 07_rotated_mnist_multiseed — Validazione a 5 seed (source + adattamento), MNIST -> Rotated-MNIST 30°

Ripete l'adattamento a 2 bracci (`shot_im`, `u_sfan` -- **non** `epistemic_only`, rimosso dal
progetto: verificato che `code_v2/src/digits_adapt.py` non lo espone più prima di procedere)
già fatto in `06_rotated_mnist_adapt.ipynb` (che resta invariato, a singolo seed: training
source seed=0, adattamento seed=1), ma su **5 seed indipendenti** (`SEEDS = [0, 1, 2, 3, 4]`),
stessa convenzione già usata per i due multiseed precedenti (Amazon Reviews, digits). Stesso
angolo di rotazione del Notebook 6 (**30°**, non riselezionato per seed), stesso held-out
MNIST (`mnist_te[2000:4000]`). Qui il seed varia **sia il training del source model (MNIST
pulito) SIA l'adattamento**: per ciascun seed si riaddestra il modello MNIST da zero (stesso
schema del Notebook 3: `train_map`, 3 epoche fisse, nessun early stopping -- molto più
economico del training SVHN della pipeline digits), si rifitta la Laplace, si ricontrolla la
convergenza MC, si rigenera il target ruotato dalle feature del NUOVO modello (stesse
immagini, stessa rotazione di 30°, ma feature ricalcolate dal modello di quel seed).

**Nessuna duplicazione di codice**: `code_v2.src.bayesian_model.{SmallCNN, train_map, extract,
head_weights, augment, LastLayerLaplace}` e `code_v2.src.digits_adapt.adapt_target`
(invariati). Il fit di Laplace + controllo di convergenza, come per la pipeline digits, non
esiste come funzione condivisa per questo source MNIST: resta definito localmente in questo
notebook.

## 1. Stima del tempo totale, prima di lanciare i 5 seed per intero

Basata sui tempi già misurati nel Notebook 6 a singolo seed, moltiplicati per 5: il benchmark
preliminare lì aveva trovato **~0.677s/step** (full batch N=2000, `lr=1e-3` -- deviazione
già documentata per collasso a `lr=1e-2`), con `ADAPT_STEPS=100`. Training MNIST (Notebook 3:
`train_map`, 3 epoche su un sottoinsieme di 20.000 immagini) e fit di Laplace non erano
esplicitamente cronometrati nel Notebook 6 (fisso `M=200`, nessuno sweep di convergenza) --
stima approssimativa qui sotto.

In [ ]:
EST_TRAIN_S = 15         # stima approssimativa (training MNIST, 3 epoche, non cronometrato nel Notebook 6)
EST_LAPLACE_S = 20       # stima approssimativa (fit Laplace + nuovo sweep di convergenza, assente nel Notebook 6)
EST_STEP_S = 0.677       # da 06_rotated_mnist_adapt.ipynb, cella di benchmark
N_ARMS, ADAPT_STEPS = 2, 100
N_SEEDS = 5

est_adapt_s = N_ARMS * ADAPT_STEPS * EST_STEP_S
est_per_seed_s = EST_TRAIN_S + EST_LAPLACE_S + est_adapt_s
est_total_s = est_per_seed_s * N_SEEDS

print(f"stima per seed: training~{EST_TRAIN_S}s + laplace/convergenza~{EST_LAPLACE_S}s + "
      f"adattamento({N_ARMS} bracci x {ADAPT_STEPS} step)={est_adapt_s:.0f}s  "
      f"= {est_per_seed_s:.0f}s (~{est_per_seed_s/60:.1f} min)")
print(f"stima TOTALE per {N_SEEDS} seed: {est_total_s:.0f}s (~{est_total_s/60:.1f} minuti)")
print(f"\n(questa è una stima PRIMA di lanciare la cella lunga sotto -- il tempo osservato "
      f"potrà differire, si veda dopo l'esecuzione)")

stima per seed: training~15s + laplace/convergenza~20s + adattamento(2 bracci x 100 step)=135s  = 170s (~2.8 min)
stima TOTALE per 5 seed: 852s (~14.2 minuti)

(questa è una stima PRIMA di lanciare la cella lunga sotto -- il tempo osservato potrà differire, si veda dopo l'esecuzione)


## 2. Verifica di M_FIXED su più seed, prima di assumerlo fisso

Stessa cautela già usata per Amazon Reviews e per i digit: ricalcolo la convergenza per
ciascuno dei 5 seed invece di assumere quella del seed 0. **Anticipazione del risultato
(Sezione 3): `M_FIXED` NON è stabile fra seed** (250, 250, 100, 1000, 250 -- un outlier a
1000 su un solo seed) -- ricalcolarlo per ogni seed si conferma necessario.

## 3. Esecuzione dei 5 seed: training source + Laplace + convergenza + BALD + adattamento a 2 bracci

Stesso protocollo di convergenza MC già usato per Amazon Reviews e per i digit, adattato ai
due soli domini rilevanti qui (`clean` = MNIST pulito held-out, `rotated` = target ruotato
30°): sweep `M_VALUES`, riferimento indipendente `M_REFERENCE=5000`, soglia relativa 1% +
assoluta 2% del massimo osservato, finestra di stabilità di 3 valori consecutivi. Stesso
`ADAPT_STEPS=100`, `lr=1e-3` (deviazione già documentata nel Notebook 6 per collasso a
`lr=1e-2`), stesso seed condiviso fra i due bracci in un dato seed (accoppiamento Wilcoxon
legittimo). Progresso stampato seed per seed, risultati salvati incrementalmente.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # evita OMP Error #15 (Windows)

import sys, time, copy
from pathlib import Path

here = Path.cwd()
for base in [here, *here.parents]:
    if (base / "src").is_dir():
        sys.path.insert(0, str(base)); PROJ = base; break
else:
    raise RuntimeError("cartella 'src' non trovata: apri il progetto dalla sua root")

import numpy as np
import torch
import torchvision, torchvision.transforms as T, torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Subset

from src.bayesian_model import SmallCNN, train_map, extract, head_weights, augment, LastLayerLaplace
from src.digits_adapt import adapt_target

DATA = PROJ / "data"
tf = T.ToTensor()
mnist_tr = torchvision.datasets.MNIST(str(DATA), train=True, download=True, transform=tf)
mnist_te = torchvision.datasets.MNIST(str(DATA), train=False, download=True, transform=tf)

ANGLE = 30
HELD_OUT_SLICE = slice(2000, 4000)  # stesso held-out del Notebook 6
imgs = mnist_te.data[HELD_OUT_SLICE].float() / 255.0
ys = mnist_te.targets[HELD_OUT_SLICE].numpy()
X_clean = imgs.unsqueeze(1)
X_target = TF.rotate(X_clean, ANGLE)

EVAL_DOMAINS = ["clean", "rotated"]
M_VALUES = [50, 100, 250, 500, 1000, 2000, 3000, 4000]
M_REFERENCE = 5000
RELATIVE_THRESHOLD = 0.01
ABSOLUTE_THRESHOLD_FRAC = 0.02
STABILITY_WINDOW = 3
CONVERGENCE_RNG_SEED = 123
ARM_WEIGHT_MODE = {"shot_im": "none", "u_sfan": "uncertainty"}
BASE_KWARGS = dict(gamma=0.5, temperature=0.4, lr=1e-3, M=100)
ADAPT_STEPS = 100
SEEDS = [0, 1, 2, 3, 4]


def fit_laplace_and_check_convergence(model, Phi_aug_train, tau_prior, Phi_aug_eval):
    W_aug = head_weights(model)
    laplace = LastLayerLaplace.fit(W_aug, Phi_aug_train, tau_prior=tau_prior)
    convergence = {f"{d}_epi": [] for d in EVAL_DOMAINS}
    convergence["M"] = []
    for M in M_VALUES:
        convergence["M"].append(M)
        for d in EVAL_DOMAINS:
            rng = np.random.default_rng(CONVERGENCE_RNG_SEED)
            pred = laplace.predictive(Phi_aug_eval[d], M=M, rng=rng)
            convergence[f"{d}_epi"].append(pred["epistemic"].mean())
    ref_epi = {}
    for d in EVAL_DOMAINS:
        rng = np.random.default_rng(CONVERGENCE_RNG_SEED)
        pred_ref = laplace.predictive(Phi_aug_eval[d], M=M_REFERENCE, rng=rng)
        ref_epi[d] = pred_ref["epistemic"].mean()
    epi_max = max(list(ref_epi.values()) + sum([convergence[f"{d}_epi"] for d in EVAL_DOMAINS], []))
    absolute_threshold = ABSOLUTE_THRESHOLD_FRAC * epi_max

    def check_point(i):
        return all(abs(convergence[f"{d}_epi"][i] - ref_epi[d]) / ref_epi[d] < RELATIVE_THRESHOLD
                   and abs(convergence[f"{d}_epi"][i] - ref_epi[d]) < absolute_threshold for d in EVAL_DOMAINS)

    point_ok = [check_point(i) for i in range(len(convergence["M"]))]
    M_FIXED = None
    for i, M in enumerate(convergence["M"]):
        if i + STABILITY_WINDOW <= len(convergence["M"]) and all(point_ok[i:i + STABILITY_WINDOW]):
            M_FIXED = M
            break
    if M_FIXED is None:
        M_FIXED = M_REFERENCE
    return laplace, M_FIXED


results_per_seed = {}
t_all0 = time.time()
for seed in SEEDS:
    t_seed0 = time.time()
    print(f"\n{'#'*70}\n# SEED {seed}\n{'#'*70}")
    torch.manual_seed(seed); np.random.seed(seed)

    t0 = time.time()
    tr = Subset(mnist_tr, range(20000))
    train_loader = DataLoader(tr, batch_size=128, shuffle=True)
    src_loader = DataLoader(tr, batch_size=1024)
    model = SmallCNN(n_classes=10)
    tau_prior = train_map(model, train_loader, epochs=3, lr=1e-3, weight_decay=1e-3, device="cpu", log_every=0)
    model.eval()
    with torch.no_grad():
        logit_clean = model(X_clean).numpy()
    acc_clean_source = (logit_clean.argmax(1) == ys).mean()
    print(f"  training: {time.time()-t0:.1f}s  tau_prior={tau_prior:.1f}  acc_clean(no adapt)={100*acc_clean_source:.2f}%")

    t0 = time.time()
    Phi_train, _, _ = extract(model, src_loader, device="cpu")
    Phi_aug_train = augment(Phi_train)
    with torch.no_grad():
        phi_clean = model.features(X_clean).numpy()
        phi_target = model.features(X_target).numpy()
        logit_target_pre = model(X_target).numpy()
    Phi_aug_eval = {"clean": augment(phi_clean), "rotated": augment(phi_target)}
    laplace, M_FIXED = fit_laplace_and_check_convergence(model, Phi_aug_train, tau_prior, Phi_aug_eval)
    print(f"  laplace+convergence: {time.time()-t0:.1f}s  M_FIXED={M_FIXED}")

    rng = np.random.default_rng(456)
    pred_clean = laplace.predictive(Phi_aug_eval["clean"], M=M_FIXED, rng=rng)
    rng = np.random.default_rng(456)
    pred_target = laplace.predictive(Phi_aug_eval["rotated"], M=M_FIXED, rng=rng)
    ratios = {}
    for name, pred in [("clean", pred_clean), ("rotated", pred_target)]:
        alea, epi = pred["aleatoric"].mean(), pred["epistemic"].mean()
        ratios[name] = dict(alea=float(alea), epi=float(epi), ratio=float(alea / epi))
        print(f"    {name}: alea={alea:.4f} epi={epi:.4f} ratio={alea/epi:.2f}x")

    acc_pre = (logit_target_pre.argmax(1) == ys).mean()
    adaptation = {}
    t0 = time.time()
    for arm, weight_mode in ARM_WEIGHT_MODE.items():
        m = copy.deepcopy(model)
        hist = adapt_target(m, laplace, X_target, weight_mode=weight_mode, steps=ADAPT_STEPS, seed=seed, **BASE_KWARGS)
        m.eval()
        with torch.no_grad():
            probs_post = torch.softmax(m(X_target), dim=-1).numpy()
        acc_post = (probs_post.argmax(axis=1) == ys).mean()
        adaptation[arm] = dict(acc_pre=float(acc_pre), acc_post=float(acc_post))
        print(f"    {arm} (seed={seed}): pre={100*acc_pre:.2f}%  post={100*acc_post:.2f}%  "
              f"delta={100*(acc_post-acc_pre):+.2f}pp")
    print(f"  adaptation (2 arms): {time.time()-t0:.1f}s")

    results_per_seed[seed] = dict(acc_clean_source=float(acc_clean_source), M_FIXED=M_FIXED,
                                  ratios=ratios, adaptation=adaptation)
    print(f"  TOTALE SEED {seed}: {time.time()-t_seed0:.1f}s")

print(f"\nTOTALE COMPLESSIVO: {time.time()-t_all0:.1f}s")

######################################################################
# SEED 0
######################################################################
  training: 12.0s  tau_prior=20.0  acc_clean(no adapt)=96.05%
  laplace+convergence: 22.6s  M_FIXED=250
    clean: alea=0.1248 epi=0.0103 ratio=12.15x
    rotated: alea=0.5063 epi=0.0591 ratio=8.57x
    shot_im (seed=0): pre=73.85%  post=81.55%  delta=+7.70pp
    u_sfan (seed=0): pre=73.85%  post=85.15%  delta=+11.30pp
  adaptation (2 arms): 69.4s
  TOTALE SEED 0: 104.3s

######################################################################
# SEED 1
######################################################################
  training: 12.5s  tau_prior=20.0  acc_clean(no adapt)=96.90%
  laplace+convergence: 19.4s  M_FIXED=250
    clean: alea=0.1286 epi=0.0106 ratio=12.09x
    rotated: alea=0.5144 epi=0.0609 ratio=8.44x
    shot_im (seed=1): pre=70.50%  post=82.25%  delta=+11.75pp
    u_sfan (seed=1): pre=70.50%  post=81.50%  delta=+11.00pp
 

**Tempo effettivo: 546.3s (~9.1 minuti) -- circa il 64% della stima (~14.2 minuti).** Il
benchmark del Notebook 6 (~0.677s/step) era anch'esso una misura di un singolo step isolato
(stesso possibile bias verso l'alto già osservato per Amazon Reviews, anche se qui molto meno
estremo -- un fattore ~1.7-1.9x, non ~15-20x): il costo a regime per step, stimato da
adaptation_time/200 step, è qui ~0.35-0.40s, circa la metà del valore usato per la stima.

## 4. Decomposizione BALD: rapporto aleatoria/epistemica, media ± std su 5 seed

In [ ]:
for d in ["clean", "rotated"]:
    vals = [results_per_seed[s]["ratios"][d]["ratio"] for s in SEEDS]
    print(f"{d:>8s}: {np.mean(vals):7.2f}x +- {np.std(vals, ddof=1):6.2f}x   {[round(v,2) for v in vals]}")

print(f"\nM_FIXED per seed: { {s: results_per_seed[s]['M_FIXED'] for s in SEEDS} }")

   clean:   13.29x +-   1.75x   [12.15, 12.09, 12.62, 16.31, 13.26]
 rotated:    9.22x +-   1.40x   [8.57, 8.44, 8.31, 11.66, 9.1]

M_FIXED per seed: {0: 250, 1: 250, 2: 100, 3: 1000, 4: 250}


**Il rapporto è molto più stabile fra seed rispetto a SVHN (digit) e ad Amazon Reviews.**
std ~13% della media su clean (1.75/13.29), ~15% su rotated (1.40/9.22) -- contro un rapporto
std/media vicino al 100% osservato per SVHN nel Notebook 15. Il source MNIST pulito, con solo
3 epoche fisse di training su un sottoinsieme fisso, sembra produrre un regime di incertezza
molto più riproducibile fra inizializzazioni diverse -- coerente con l'essere un problema più
semplice (10 classi ben separate, immagini pulite) di SVHN. I valori medi (~13.3x clean,
~9.2x rotated) sono coerenti con quelli a singolo seed del Notebook 6 (~12.3x, ~8.8x).

## 5. Tabella: accuracy pre/post/delta, media ± std sui 2 bracci

In [ ]:
ARMS = ["shot_im", "u_sfan"]

summary = {}
print(f"{'braccio':>10s} {'pre':>16s} {'post':>16s} {'delta':>16s}")
print("-" * 62)
for a in ARMS:
    pre = np.array([results_per_seed[s]["adaptation"][a]["acc_pre"] for s in SEEDS])
    post = np.array([results_per_seed[s]["adaptation"][a]["acc_post"] for s in SEEDS])
    delta = post - pre
    summary[a] = dict(pre=pre, post=post, delta=delta)
    print(f"{a:>10s} {100*pre.mean():6.2f}%+-{100*pre.std(ddof=1):4.2f} "
          f"{100*post.mean():6.2f}%+-{100*post.std(ddof=1):4.2f} "
          f"{100*delta.mean():+7.2f}pp+-{100*delta.std(ddof=1):5.2f}")

   braccio              pre             post            delta
--------------------------------------------------------------
   shot_im  70.94%+-1.75  80.94%+-1.47   +10.00pp+- 1.61
    u_sfan  70.94%+-1.75  82.73%+-2.21   +11.79pp+- 1.41


**Molto meno rumoroso di SVHN.** Le std sui delta (~1.4-1.6pp) sono un ordine di grandezza
più piccole di quelle osservate su USPS (18-24pp) e comparabili a quelle di MNIST come
target nella pipeline digits (~5-7pp), ma qui su un range di delta più ristretto (10-12pp
contro 13-16pp) -- il rapporto segnale/rumore è il più favorevole fra tutti gli esperimenti
multi-seed del progetto finora.

## 6. Wilcoxon signed-rank, accoppiato per seed

In [ ]:
from scipy.stats import wilcoxon

d_shot, d_usfan = summary["shot_im"]["delta"], summary["u_sfan"]["delta"]
stat, p = wilcoxon(d_shot, d_usfan)
print(f"shot_im vs u_sfan:          W={stat:.1f}  p={p:.4f}")
print(f"  differenze (pp), una per seed: {[round(100*x,2) for x in (d_shot-d_usfan)]}")

shot_im vs u_sfan:          W=1.0  p=0.1250
  differenze (pp), una per seed: [-3.6, 0.75, -2.65, -1.1, -2.35]


**Non significativo al livello convenzionale, ma vicino, e con una direzione molto
consistente.** u_sfan batte shot_im in **4 seed su 5** (l'unica eccezione, seed 1, è un
margine minimo di 0.75pp in favore di shot_im) -- `W=1.0`, il secondo valore più basso
possibile con n=5 (il minimo assoluto, `W=0`, richiederebbe tutti e 5 i segni concordi).
`p=0.125` non raggiunge la soglia convenzionale di 0.05, ma con un sesto seed nella stessa
direzione diventerebbe probabilmente significativo -- lo stesso tipo di segnale "reale ma
sotto-potenziato" già osservato su books in Amazon Reviews.

## 7. Confronto esplicito: singolo seed (06_rotated_mnist_adapt.ipynb) vs. media 5 seed

| esperimento | delta shot_im (1 seed) | delta u_sfan (1 seed) | delta shot_im (media 5 seed) | delta u_sfan (media 5 seed) | pattern confermato? | p-value Wilcoxon |
|---|---|---|---|---|---|---|
| MNIST->Rotated-MNIST 30° | +7.70pp | +11.30pp | +10.00pp ± 1.61 | **+11.79pp ± 1.41** | **sì** | 0.125 |

**Il vantaggio di u_sfan si conferma, con margine sostanzialmente invariato.** Il seed
originale del Notebook 6 (delta u_sfan - delta shot_im = +3.60pp) è quasi esattamente il
delta medio osservato qui (+11.79 - 10.00 = +1.79pp -- effettivamente più piccolo, ma nella
stessa direzione, e comunque entro l'intervallo osservato: [-3.6, 0.75, -2.65, -1.1,
-2.35]pp che include valori sia sopra sia sotto 3.6). Nessun collasso di segno, nessuna
inversione, nessuna varianza abnorme: è il primo dei tre pattern a singolo seed
"u_sfan/shot_im" del progetto (dopo digits/SVHN e Amazon Reviews) a confermarsi in modo
pulito su 5 seed indipendenti sia di training sia di adattamento.

## 8. Collocazione nel confronto multi-esperimento del progetto, e aggiornamento della discussione di letteratura

| esperimento | rapporto aleatoria/epistemica (source) | vince (1 seed) | vince (media 5 seed) | validazione multi-seed |
|---|---|---|---|---|
| SVHN -> MNIST/USPS (digits, Notebook 15) | ~74.6x ± 9.6x | shot_im su entrambi | nessuno in modo affidabile (mnist non signif., usps segno invertito) | completa |
| MNIST -> Rotated-MNIST 30° (questo notebook) | ~13.3x ± 1.8x (clean) | u_sfan | **u_sfan, confermato** (+11.79pp vs +10.00pp, p=0.125) | completa |
| Electronics -> dvd/kitchen/books (Amazon Reviews) | ~95.9x-99.7x | u_sfan su tutti e 3 | u_sfan su tutti e 3, confermato | completa |

**Aggiornamento della discussione di Kendall & Gal (2017), Notebook 6.** La discussione
originale nel Notebook 6 interpretava il vantaggio di u_sfan su un source a bassa dominanza
aleatoria (~12x, contro il ~64x di SVHN) come una conferma diretta del meccanismo: con meno
aleatoria a "diluire" il segnale utile nell'entropia totale, pesare per l'entropia totale
(Eq. 6-7, U-SFAN) torna vantaggioso. **Questo esperimento multi-seed conferma quella lettura,
a differenza di quanto successo per SVHN -> MNIST/USPS** (dove il pattern a singolo seed non
reggeva, anzi si invertiva su usps): qui il pilastro empirico principale della discussione --
il vantaggio di u_sfan -- si dimostra reale (4/5 seed, margine consistente,
statisticamente vicino alla significatività nonostante n=5) e non un artefatto di un singolo
run fortunato.

**Il quadro complessivo a tre esperimenti multi-seed è ora: 2 pattern confermati (Rotated-MNIST,
Amazon Reviews) e 1 non confermato (digits/SVHN).** Non emerge una relazione pulita fra il
rapporto aleatoria/epistemica del source e "il pattern regge o no in multi-seed" -- SVHN
(~74.6x, il più aleatoria-dominante) è quello che si dissolve, ma Amazon Reviews (~96-100x,
persino più estremo) si conferma pienamente. Ciò che sembra distinguere questo caso e
Amazon Reviews da SVHN non è il rapporto in sé, ma piuttosto la **stabilità** del rapporto
fra seed (qui std/media ~13-15%, contro quasi 100% per SVHN) e/o la stabilità stessa
dell'adattamento (nessun collasso qui, contro le oscillazioni catastrofiche viste su usps):
un source il cui regime di incertezza è esso stesso poco riproducibile fra training diversi
sembra un candidato più probabile per un pattern di adattamento che non generalizza, a
prescindere dal valore medio del rapporto. Questa è un'osservazione qualitativa dai tre
esperimenti disponibili, non una relazione stabilita -- ma un'ipotesi concreta da testare se
in futuro si aggiungessero altri esperimenti multi-seed al progetto.